# Hands-On 8: Imbalance, weights and thresholds

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from mlcourse.labs import make_imbalance_data, split_classification, binary_metrics
from mlcourse.widgets import interactive_threshold

## 1. Inspect the class distribution

Generate the two-feature dataset and create a stratified split.

**Prediction:** Could a model obtain high accuracy while predicting only class `0`?

*Your response.*

In [ ]:
X, y = make_imbalance_data()
X_train, X_test, y_train, y_test = split_classification(X, y)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
y.value_counts().sort_index().plot.bar(ax=axes[0], color=['#0072B2', '#D55E00'], rot=0)
axes[0].set(xlabel='Class', ylabel='Count', title='Class distribution')
for value, marker, color in [(0, 'o', '#0072B2'), (1, '^', '#D55E00')]:
    axes[1].scatter(X.loc[y == value, 'A1'], X.loc[y == value, 'A2'], marker=marker, color=color, s=24, label=str(value))
axes[1].set(xlabel='A1', ylabel='A2', title='Feature space')
axes[1].legend(title='Class')
plt.show()
display(pd.DataFrame({'train_count': y_train.value_counts(), 'test_count': y_test.value_counts()}))

**Observation:** Compare class counts in both partitions.

*Your response.*

**Explanation:** Why does stratification help make the comparison interpretable?

*Your response.*

## 2. Establish a simple baseline

Compare an always-majority prediction with a depth-two decision tree.

**Prediction:** Which metric will expose the majority classifier most clearly?

*Your response.*

In [ ]:
shallow_tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
results = pd.DataFrame({
    'Always majority': binary_metrics(y_test, np.zeros(len(y_test), dtype=int)),
    'Depth-two tree': binary_metrics(y_test, shallow_tree.predict(X_test)),
}).T
display(results)

**Observation:** Compare accuracy with minority recall.

*Your response.*

**Explanation:** Explain why these two metrics can disagree.

*Your response.*

## 3. Compare standard models

Fit logistic regression and k-NN on the same split. Add their metrics to the table.

**Prediction:** Does a more flexible boundary necessarily improve minority recall?

*Your response.*

In [ ]:
models = {'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression()),
          'k-NN': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))}
for name, model in models.items():
    model.fit(X_train, y_train)
    results.loc[name] = binary_metrics(y_test, model.predict(X_test))
display(results)

**Observation:** Which model has the highest recall, and what precision accompanies it?

*Your response.*

**Explanation:** Explain the trade-off using false positives and false negatives.

*Your response.*

## 4. Change class weights

Fit logistic regression with `class_weight="balanced"`.

**Prediction:** How should stronger minority weighting change the positive region?

*Your response.*

In [ ]:
balanced_model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced'))
balanced_model.fit(X_train, y_train)
results.loc['Weighted logistic regression'] = binary_metrics(y_test, balanced_model.predict(X_test))
display(results.loc[['Logistic regression', 'Weighted logistic regression']])

**Observation:** Compare recall and precision before and after weighting.

*Your response.*

**Explanation:** Why does weighting affect the fitted boundary?

*Your response.*

## 5. Move the decision threshold

Lower the threshold from `0.500` to `0.200`; then compare uniform and balanced weights.

**Prediction:** What will happen to predicted positives when the threshold decreases?

*Your response.*

In [ ]:
threshold_lab = interactive_threshold(X_train, X_test, y_train, y_test)
display(threshold_lab.widget)

**Observation:** Record the change in the boundary, false negatives and false positives.

*Your response.*

**Explanation:** Distinguish changing a threshold from changing the fitted model.

*Your response.*